### DB日報資料撈取

In [3]:
import pandas as pd
import numpy as np
import pyodbc
import configparser

In [ ]:
cfg = configparser.ConfigParser()
cfg.read(r"D:\yujui\SqlServer18.ini")      # ← 指定絕對路徑
print(cfg.sections())                       # 應該會看到 ['mssql']

if 'mssql' not in cfg:
    raise RuntimeError("找不到 [mssql] 區段，請檢查 ini 檔位置與內容")

sec = cfg['mssql']
database = "DSC_CRM_SP"   # 寫在程式內

conn = pyodbc.connect(
    f"DRIVER={{SQL Server}};"
    f"SERVER={sec['server']};DATABASE={database};"
    f"UID={sec['uid']};PWD={sec['pwd']};",
    autocommit=True
)

['mssql']


In [ ]:
query = """
select GY001,GY003,GY004,GY022,GY012,CREATE_DATE
,ROW_NUMBER() OVER(PARTITION BY GY001,GY003,GY004,GY012 ORDER BY CREATE_DATE,GY022)
from CRMGY
"""
LeadInfo = pd.read_sql(query, conn)
LeadInfo

C:\Users\DSC\AppData\Local\Temp\ipykernel_27352\3919206088.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  LeadInfo = pd.read_sql(query, conn)


,LD005,LD010
0,0000000001,致電03 452 2102，電話已是空號..查無其他有效電話...且google顯示暫停營業
1,0000000016,當初導入失敗，user認為鼎新系統卡控太多，\n目前為淡季生意很差，所以也不會購買任何系統
2,0000000016,OT探詢
3,0000000016,能管探詢
4,0000000017,確認ERP與異質系統整合需求
...,...,...
756984,Y000000000,回撥線索。手機一樣是暫停使用，mail也無回應@@故先結案處理。
756985,Y000000000,軍方走招標
756986,Y000000000,9/17 16:27去電曾R，掛斷電話轉語音信箱。
756987,Y000000000,2025-12-22 11:04無人接聽


上傳資料至BigQuery

In [4]:
import os
import configparser
import google.generativeai as genai
import pandas_gbq
import time

d:\miniforge3\envs\yujui_even\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
d:\miniforge3\envs\yujui_even\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\DSC\AppData\Local\Temp\ipykernel_23948\3795072751.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://gith

In [5]:
# 先從 ini 檔讀（可放在任意路徑），如果沒有就退而使用環境變數
cfg = configparser.ConfigParser()
cfg.read(r"D:\yujui\GoogleCloud.ini")   # 自行調整路徑

service_account_key = os.getenv(
    "GOOGLE_APPLICATION_CREDENTIALS",
    cfg.get("gcp", "service_account_key", fallback=""),
)
if not service_account_key:
    raise RuntimeError("請先設定 GOOGLE_APPLICATION_CREDENTIALS 環境變數或在 gcp.ini 裡填 service_account_key")

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = service_account_key

PROJECT_ID = os.getenv(
    "PROJECT_ID",
    cfg.get("gcp", "project_id", fallback="digiwin-ai"),
)
DATASET_ID = os.getenv(
    "DATASET_ID",
    cfg.get("gcp", "dataset_id", fallback="digiwin_data_analysis"),
)
GEMINI_API_KEY = os.getenv(
    "GEMINI_API_KEY",
    cfg.get("gcp", "gemini_api_key", fallback=""),
)

# 唯一還是硬寫的就是 model 名稱
model = genai.GenerativeModel("gemini-2.5-flash")

In [18]:
# 列出所有可用的模型名稱
print("--- 你目前可用的模型清單 ---")
available_models = []
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f"可用模型: {m.name}")
        available_models.append(m.name)

# 自動挑選一個最適合的 (優先找 flash, 再找 pro)
best_model = None
for name in available_models:
    if 'gemini-2.5-flash' in name:
        best_model = name
        break
if not best_model:
    for name in available_models:
        if 'gemini-pro' in name:
            best_model = name
            break

print(f"\n✅ 系統建議使用的模型名稱: {best_model}")

--- 你目前可用的模型清單 ---
可用模型: models/gemini-2.5-flash
可用模型: models/gemini-2.5-pro
可用模型: models/gemini-2.0-flash
可用模型: models/gemini-2.0-flash-001
可用模型: models/gemini-2.0-flash-exp-image-generation
可用模型: models/gemini-2.0-flash-lite-001
可用模型: models/gemini-2.0-flash-lite
可用模型: models/gemini-2.5-flash-preview-tts
可用模型: models/gemini-2.5-pro-preview-tts
可用模型: models/gemma-3-1b-it
可用模型: models/gemma-3-4b-it
可用模型: models/gemma-3-12b-it
可用模型: models/gemma-3-27b-it
可用模型: models/gemma-3n-e4b-it
可用模型: models/gemma-3n-e2b-it
可用模型: models/gemini-flash-latest
可用模型: models/gemini-flash-lite-latest
可用模型: models/gemini-pro-latest
可用模型: models/gemini-2.5-flash-lite
可用模型: models/gemini-2.5-flash-image
可用模型: models/gemini-2.5-flash-lite-preview-09-2025
可用模型: models/gemini-3-pro-preview
可用模型: models/gemini-3-flash-preview
可用模型: models/gemini-3.1-pro-preview
可用模型: models/gemini-3.1-pro-preview-customtools
可用模型: models/gemini-3.1-flash-lite-preview
可用模型: models/gemini-3-pro-image-preview
可用模型: models/nano-banan

In [21]:
# 假設前面已經有 LeadInfo 這個 pandas.DataFrame
# 且已經在開機環境中設定好 GOOGLE_APPLICATION_CREDENTIALS 等必要 env

# 先把欄位名稱裡的空白／句點換掉，BigQuery 不喜歡
lead = LeadInfo.copy()
lead.columns = [str(c).replace('.', '_').replace(' ', '_') for c in lead.columns]
print("LeadInfo 欄位：", lead.columns.tolist())

# 上傳
try:
    print("正在上傳 LeadInfo …")
    pandas_gbq.to_gbq(
        lead,
        f'{DATASET_ID}.LeadInfo',         # table name 可以自己訂
        project_id=PROJECT_ID,
        if_exists='replace'              # 視需求改成 'append'
    )
    print("上傳完成")
except Exception as e:
    print(f"❌ 上傳失敗：{e}")



LeadInfo 欄位： ['LD005', 'LD010']
正在上傳 LeadInfo …


100%|██████████| 1/1 [00:00<00:00, 999.60it/s]

上傳完成


In [6]:
# （可選）讀回來確認
lead_df = pandas_gbq.read_gbq(
    f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.LeadInfo`",
    project_id=PROJECT_ID
)
lead_df.head()

Downloading: 100%|██████████|


,LD005,LD010
0,0000000001,致電03 452 2102，電話已是空號..查無其他有效電話...且google顯示暫停營業
1,0000000016,當初導入失敗，user認為鼎新系統卡控太多，\n目前為淡季生意很差，所以也不會購買任何系統
2,0000000016,OT探詢
3,0000000016,能管探詢
4,0000000017,確認ERP與異質系統整合需求


In [9]:
import os
import re
import pandas as pd
import json
import csv
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# ---------- JSON 清理 + 欄位保證 ----------
def clean_json(text: str) -> str:
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        text = match.group(0)
    text = text.replace("True", "true").replace("False", "false")
    return text

def ensure_stage_fields(result: dict) -> dict:
    return {
        "E": result.get("E", False),
        "D": result.get("D", False),
        "C2": result.get("C2", False),
        "C1": result.get("C1", False),
        "B": result.get("B", False),
        "A": result.get("A", False),
        "stage_group": result.get("stage_group", "none")
    }

# ---------- LLM 呼叫（新版 Prompt） ----------
def quick_stage_check(log_text: str) -> dict:
    prompt = f"""
    你是一個快速審查器，請根據以下日誌內容，判斷它是否處於以下六個階段之一：

    - E: 潛客售前規劃與研究、電話開發非業務責任區名單
    - D: 引發客戶潛在窗口興趣、感受痛苦但不知問題點、設想客戶可能的痛點
    - C2: 知道問題點但不知道解決方案、拜訪客戶窗口或權利者、探詢痛點與問題、確認客戶認同痛點與問題、判斷可能商機、蒐集與整理客戶認同的商業效益、知道有解決方案未形成共識、擴大痛點、提出初步解決方案
    - C1: 形成部分共識、蒐集資料、拜訪決策者、形成內部共識/決定需求要件、其他廠商參與評估、了解客戶目標效益、探詢其他痛點、判斷其他商機
    - B: 擬定競爭策略、發展客戶增值計畫、展示(DEMO)、評估方案、報價、簡報、考慮價錢/風險、排除疑慮、要求客戶承諾
    - A: 選定廠商與解決方案、議價與議約、正式簽約

    請只輸出 JSON 格式，不要加任何解釋或文字。
    JSON 格式必須完全符合以下結構，且只能包含這七個欄位：
    {{
      "E": true/false,
      "D": true/false,
      "C2": true/false,
      "C1": true/false,
      "B": true/false,
      "A": true/false,
      "stage_group": "成交前/成交後/none"
    }}

    判斷規則：
    1. 僅當日誌明確描述「流程階段」時才標記為 true。
    2. 如果日誌只是聯繫、描述背景、提供資訊、表達意見或純粹抱怨，則所有欄位必須是 false，stage_group = "none"。
    3. 如果日誌描述屬於 E / D / C2 / C1 / B，則 stage_group = "成交前"。
    4. 如果日誌描述屬於 A，則 stage_group = "成交後"。
    5. 不允許模糊推測，必須有明確語意才算 true。
    6. 請只輸出 JSON，不要加任何解釋或文字。

    日誌內容：{log_text}

    請直接輸出 JSON：
    """
    response = model.generate_content(prompt)
    try:
        cleaned = clean_json(response.text)
        parsed = json.loads(cleaned)
        return ensure_stage_fields(parsed)
    except Exception:
        return {"E": False, "D": False, "C2": False, "C1": False, "B": False, "A": False, "stage_group": "none"}

# ---------- 並行化 Stage Check ----------
def parallel_stage_check(batch: pd.DataFrame, max_workers=10) -> pd.DataFrame:
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(lambda row: quick_stage_check(row["LD010"]), batch.to_dict("records")))
    return pd.DataFrame(results, index=batch.index)



In [11]:
import os
import re
import pandas as pd
import json
import csv
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# ---------- JSON 清理 + 欄位保證 ----------
def clean_json(text: str) -> str:
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        text = match.group(0)
    text = text.replace("True", "true").replace("False", "false")
    return text

def ensure_stage_fields(result: dict) -> dict:
    return {
        "E": result.get("E", False),
        "D": result.get("D", False),
        "C2": result.get("C2", False),
        "C1": result.get("C1", False),
        "B": result.get("B", False),
        "A": result.get("A", False),
        "stage_group": result.get("stage_group", "none")
    }

# ---------- LLM 呼叫（新版 Prompt） ----------
def quick_stage_check(log_text: str) -> dict:
    prompt = f"""
    你是一個快速審查器，請根據以下日誌內容，判斷它是否處於以下六個階段之一：

    - E (公田客戶)：業務認定客戶有潛在機會。 客戶處於「感受痛苦但不知問題點」的階段。 業務行為以「長期經營」為主。
    - D：業務認定客戶有明確痛點。 客戶處於「知道問題但不知有解決方案」的階段。 業務正處於需求探訪，並準備尋找權力人士進行提案。
    - C2：業務能清楚指出痛點，且客戶認同或準備擴大痛點。 客戶處於「知道可解決但內部未形成共識」的階段。 業務行為重點在於「促發需求」。
    - C1 (專案立案)：專案正式成立。 業務已確認對接窗口、成案要件，並與負責人確認後續時程。 客戶開始形成部分共識並收集相關資料。
    - C1 -> B (提供提案)：業務針對方案效益進行提案報告，展示方案能滿足核心問題或優於競爭對手。 客戶正尋找相關廠商並設想解決問題的要件。
    - B (專案定案)：專案進入競爭中的 DEMO 階段，或進行高階主管簡報。 客戶處於「決定需求」並開始「評估方案」的購買階段。
    - A (專案決案)：進入最終風險疑慮排除、議價與議約階段，且當月具備簽約可能。 客戶處於「評估風險」並選定廠商的最後決策期。

    請只輸出 JSON 格式，不要加任何解釋或文字。
    JSON 格式必須完全符合以下結構，且只能包含這七個欄位：
    {{
      "E": true/false,
      "D": true/false,
      "C2": true/false,
      "C1": true/false,
      "B": true/false,
      "A": true/false,
      "stage_group": "成交前/成交後/none"
    }}

    判斷規則：
    1. 僅當日誌明確描述「流程階段」時才標記為 true。
    2. 如果日誌只是聯繫、描述背景、提供資訊、表達意見或純粹抱怨，則所有欄位必須是 false，stage_group = "none"。
    3. 如果日誌描述屬於 E / D / C2 / C1 / B，則 stage_group = "成交前"。
    4. 如果日誌描述屬於 A，則 stage_group = "成交後"。
    5. 不允許模糊推測，必須有明確語意才算 true。
    6. 請只輸出 JSON，不要加任何解釋或文字。

    日誌內容：{log_text}

    請直接輸出 JSON：
    """
    response = model.generate_content(prompt)
    try:
        cleaned = clean_json(response.text)
        parsed = json.loads(cleaned)
        return ensure_stage_fields(parsed)
    except Exception:
        return {"E": False, "D": False, "C2": False, "C1": False, "B": False, "A": False, "stage_group": "none"}

# ---------- 並行化 Stage Check ----------
def parallel_stage_check(batch: pd.DataFrame, max_workers=10) -> pd.DataFrame:
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(lambda row: quick_stage_check(row["LD010"]), batch.to_dict("records")))
    return pd.DataFrame(results, index=batch.index)

# ---------------- 單批次測試 ----------------
batch_size = 1000
sample_df = lead_df.sample(batch_size, random_state=42).copy()

# 清理換行符號，避免 CSV 跳行
sample_df["LD010"] = sample_df["LD010"].astype(str).str.replace("\n", " ", regex=False)

# 並行判斷六個階段
batch_results = parallel_stage_check(sample_df, max_workers=10)

# 合併結果
batch = pd.concat([sample_df, batch_results], axis=1)

# 檢查筆數是否一致
assert len(batch) == len(sample_df), "❌ 合併後筆數不一致，請檢查 parallel_stage_check"

# 顯示前 20 筆結果
print(batch.head(20))

# 🚫 不上傳 BigQuery，改成匯出 CSV
output_file = "test_ma.csv"
batch.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL   # 所有欄位加引號，避免跳行
)
print(f"✅ 測試批次已匯出到 {output_file}")

             LD005                                              LD010      E  \
233174  0000164161  行政單位小姐說目前跟德安配合不到五年，人事才剛在測試中，他說老實講用的沒有很好，但財會跟採購...  False   
623359  8200457707                    私田分流名單，吳小姐表示目的系統穩定使用中， 不需要資料，謝謝  False   
333038  0000671839  尚使用EGO(進銷存=採購二位/ 會計一位)，\r 客戶告知硬體無法升等，直接導向12/6說...  False   
326520  0000669641                                                 客情  False   
590947  8200419248                                             SM版更報價  False   
114422  0000128029  今天拜訪王先生做初步的認識，以下是會議重點:(1)目前的射出機臺因爲有專屬性的關係導致機器之...   True   
321078  0000667192                                             ｈｒ小版問題  False   
437408  0000701886  新設名單，找今年評估系統窗口何先生，何先生聽到鼎新說目前不需要哦，原定探詢資安後續評估狀況，...  False   
315192  0000664829  主要產品：學生用品、繡花、馬術用品買賣，目前用易品，戴先生表示易品的伺服器本身不穩定，容易當...  False   
521220  8101422383  聯絡王老闆接電表示會計王小姐是她女兒，因為廠內空氣不好所以他都叫她在家裡幫忙做帳，關於系統的...   True   
729514  8200539819  葉先生今天有出差會議沒辦法進公司，但可以發信先詢問她會看，改詢問盧小姐當初主要是看到ISO文...   True   
497732  8101277268  一開始說周小姐去銀行了，跟黎先生講到一半

# ---------------- 主流程 ----------------
chunk_size = 50000  # 🚀 大批次
total = len(lead_df)
start_point = get_last_progress()

for start in tqdm(range(start_point, total, chunk_size), desc="處理進度", unit="筆", ncols=100):
    batch = lead_df.iloc[start:start+chunk_size].copy()
    batch_results = parallel_stage_check(batch, max_workers=10)  # 🚀 並行化
    batch = pd.concat([batch, batch_results], axis=1)
    batch = apply_decision_rules(batch)

    success = upload_with_retry(batch, f"{DATASET_ID}.LeadInfoProcessed")
    if success:
        save_progress(start)
        
# 候補跑失敗批次（並行化 + 進度條 + 自動重試 + 詳細錯誤紀錄）
rerun_failed_batches_parallel(f"{DATASET_ID}.LeadInfoProcessed")

In [12]:
# ← 將結果下載至本地
output_path = r"D:\yujui\痛點需求地圖\LeadInfo_step1.csv"
lead_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"✓ 結果已導出至：{output_path}")

✓ 結果已導出至：D:\yujui\痛點需求地圖\LeadInfo_step1.csv


In [10]:
# ---------------- 單批次測試 ----------------
batch_size = 1000
sample_df = lead_df.sample(batch_size, random_state=42).copy()

# 清理換行符號，避免 CSV 跳行
sample_df["LD010"] = sample_df["LD010"].astype(str).str.replace("\n", " ", regex=False)

# 並行判斷六個階段
batch_results = parallel_stage_check(sample_df, max_workers=10)

# 合併結果
batch = pd.concat([sample_df, batch_results], axis=1)

# 檢查筆數是否一致
assert len(batch) == len(sample_df), "❌ 合併後筆數不一致，請檢查 parallel_stage_check"

# 顯示前 20 筆結果
print(batch.head(20))

# 🚫 不上傳 BigQuery，改成匯出 CSV
output_file = "test_batch.csv"
batch.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL   # 所有欄位加引號，避免跳行
)
print(f"✅ 測試批次已匯出到 {output_file}")


             LD005                                              LD010      E  \
233174  0000164161  行政單位小姐說目前跟德安配合不到五年，人事才剛在測試中，他說老實講用的沒有很好，但財會跟採購...  False   
623359  8200457707                    私田分流名單，吳小姐表示目的系統穩定使用中， 不需要資料，謝謝  False   
333038  0000671839  尚使用EGO(進銷存=採購二位/ 會計一位)，\r 客戶告知硬體無法升等，直接導向12/6說...  False   
326520  0000669641                                                 客情  False   
590947  8200419248                                             SM版更報價  False   
114422  0000128029  今天拜訪王先生做初步的認識，以下是會議重點:(1)目前的射出機臺因爲有專屬性的關係導致機器之...  False   
321078  0000667192                                             ｈｒ小版問題  False   
437408  0000701886  新設名單，找今年評估系統窗口何先生，何先生聽到鼎新說目前不需要哦，原定探詢資安後續評估狀況，...   True   
315192  0000664829  主要產品：學生用品、繡花、馬術用品買賣，目前用易品，戴先生表示易品的伺服器本身不穩定，容易當...  False   
521220  8101422383  聯絡王老闆接電表示會計王小姐是她女兒，因為廠內空氣不好所以他都叫她在家裡幫忙做帳，關於系統的...  False   
729514  8200539819  葉先生今天有出差會議沒辦法進公司，但可以發信先詢問她會看，改詢問盧小姐當初主要是看到ISO文...  False   
497732  8101277268  一開始說周小姐去銀行了，跟黎先生講到一半

## 主流程：全量商機等級判定（756,989 筆）

斷點續傳 + CSV 輸出，不依賴 BigQuery。

In [ ]:
import json, time, csv as csv_mod
from pathlib import Path

OUTPUT_FILE   = r"D:\yujui\痛點需求地圖\lead_stage_results.csv"
PROGRESS_FILE = r"D:\yujui\痛點需求地圖\lead_stage_progress.json"

def get_last_progress() -> int:
    if Path(PROGRESS_FILE).exists():
        return json.loads(Path(PROGRESS_FILE).read_text()).get("last_start", 0)
    return 0

def save_progress(start: int):
    Path(PROGRESS_FILE).write_text(json.dumps({"last_start": start}))

def apply_decision_rules(df: pd.DataFrame) -> pd.DataFrame:
    """每筆日誌取最高銷售階段作為代表（A > B > C1 > C2 > D > E > none）"""
    stage_order = ["A", "B", "C1", "C2", "D", "E"]
    def top_stage(row):
        for s in stage_order:
            if row.get(s, False):
                return s
        return "none"
    df = df.copy()
    df["top_stage"] = df.apply(top_stage, axis=1)
    return df

print("✅ 輔助函式載入完成")
print(f"   輸出檔：{OUTPUT_FILE}")
print(f"   進度檔：{PROGRESS_FILE}")

In [ ]:
# ────────────────────────────────────────────────────────────────
# 主流程：全量商機等級判定（756,989 筆），斷點續傳，輸出 CSV
# ────────────────────────────────────────────────────────────────
CHUNK_SIZE  = 5000   # 每批 5000 筆（保守，避免 OOM）
MAX_WORKERS = 6      # ThreadPoolExecutor 並發數

total       = len(lead_df)
start_point = get_last_progress()
file_exists = Path(OUTPUT_FILE).exists() and start_point > 0

print(f"總筆數：{total:,}  |  從第 {start_point:,} 筆繼續  |  預計批次：{(total - start_point) // CHUNK_SIZE + 1}")

for start in tqdm(range(start_point, total, CHUNK_SIZE), desc="商機等級", unit="批", ncols=100):
    batch = lead_df.iloc[start : start + CHUNK_SIZE].copy()
    batch["LD010"] = batch["LD010"].astype(str).str.replace("\n", " ", regex=False).str.replace("\r", " ", regex=False)

    # LLM 並行判斷
    batch_results = parallel_stage_check(batch, max_workers=MAX_WORKERS)
    batch = pd.concat([batch, batch_results], axis=1)

    # 取最高階段
    batch = apply_decision_rules(batch)

    # 追加寫入 CSV
    batch.to_csv(
        OUTPUT_FILE,
        mode="a",
        index=False,
        header=not file_exists,
        encoding="utf-8-sig",
        quoting=csv_mod.QUOTE_ALL,
    )
    file_exists = True

    # 存進度（下次從 start + CHUNK_SIZE 繼續）
    save_progress(start + CHUNK_SIZE)

    time.sleep(0.5)  # 輕微緩衝，避免 RPM 觸頂

print(f"\n✅ 全量完成！結果存於：{OUTPUT_FILE}")

# 快速統計
result_df = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig")
print(f"\n📊 top_stage 分布（共 {len(result_df):,} 筆）：")
print(result_df["top_stage"].value_counts().to_string())